# MountainCar-v0

## Environment Overview

MountainCar is a classic control problem where a car is on a one-dimensional track, positioned between two "mountains". The goal is to drive up the mountain on the right; however, the car's engine is not strong enough to scale the mountain in a single pass. Therefore, the only way to succeed is to drive back and forth to build up momentum.

## Observation Space

The observation is a 2-dimensional continuous array representing:

1. **Car Position** (x): Position along the x-axis (range: -1.2 to 0.6)
   - -1.2 is the leftmost point
   - 0.5 is the rightmost point (goal)
2. **Car Velocity** (v): Velocity of the car (range: -0.07 to 0.07)

## Action Space

The action space is discrete with 3 possible actions:
- 0: Accelerate to the left
- 1: Don't accelerate
- 2: Accelerate to the right

## Reward Structure

- Reward of -1 for each time step until the car reaches the flag (position ≥ 0.5)
- Episode terminates when the car reaches the flag or after 200 steps
- Goal: Minimize the number of steps to reach the flag

## Success Condition

The episode is successful when the car reaches position ≥ 0.5 on the right side of the mountain.


In [ ]:
import gymnasium as gym
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict
import time

## Simple Momentum-Based Policy

This policy uses the car's position and velocity to decide actions based on momentum building.

In [ ]:
def momentum_policy(observation):
    """
    Simple policy that builds momentum by going in the direction of velocity
    """
    position, velocity = observation
    
    # If moving right and have enough momentum, keep going right
    if velocity > 0:
        return 2  # Accelerate right
    # If moving left, keep going left to build momentum
    elif velocity < 0:
        return 0  # Accelerate left
    # If not moving, go right (towards goal)
    else:
        return 2  # Accelerate right

def test_policy(policy, num_episodes=10, render_mode=None, max_steps=200):
    """
    Test a policy on MountainCar environment
    """
    env = gym.make('MountainCar-v0', render_mode=render_mode)
    
    total_rewards = []
    episode_lengths = []
    successes = 0
    
    for episode in range(num_episodes):
        observation, info = env.reset()
        total_reward = 0
        
        for step in range(max_steps):
            action = policy(observation)
            observation, reward, terminated, truncated, info = env.step(action)
            total_reward += reward
            
            if terminated or truncated:
                if observation[0] >= 0.5:  # Success condition
                    successes += 1
                break
        
        total_rewards.append(total_reward)
        episode_lengths.append(step + 1)
        
        if num_episodes <= 5:
            print(f"Episode {episode + 1}: Reward = {total_reward}, Steps = {step + 1}, Success = {observation[0] >= 0.5}")
    
    env.close()
    
    avg_reward = np.mean(total_rewards)
    avg_steps = np.mean(episode_lengths)
    success_rate = successes / num_episodes
    
    print(f"\nResults over {num_episodes} episodes:")
    print(f"Average reward: {avg_reward:.2f}")
    print(f"Average steps: {avg_steps:.2f}")
    print(f"Success rate: {success_rate:.2%}")
    
    return avg_reward, avg_steps, success_rate

In [ ]:
# Test the momentum policy
print("Testing Momentum Policy:")
test_policy(momentum_policy, num_episodes=5)

## Q-Learning Solution

Implementing a Q-learning agent that discretizes the continuous state space and learns optimal actions.

In [ ]:
class QLearningAgent:
    def __init__(self, n_actions=3, learning_rate=0.1, discount_factor=0.99, 
                 epsilon=1.0, epsilon_decay=0.995, min_epsilon=0.01,
                 position_bins=20, velocity_bins=20):
        self.n_actions = n_actions
        self.lr = learning_rate
        self.gamma = discount_factor
        self.epsilon = epsilon
        self.epsilon_decay = epsilon_decay
        self.min_epsilon = min_epsilon
        
        # State space discretization
        self.position_bins = position_bins
        self.velocity_bins = velocity_bins
        
        # Position range: [-1.2, 0.6], Velocity range: [-0.07, 0.07]
        self.position_space = np.linspace(-1.2, 0.6, position_bins)
        self.velocity_space = np.linspace(-0.07, 0.07, velocity_bins)
        
        # Initialize Q-table
        self.q_table = defaultdict(lambda: np.zeros(n_actions))
        
    def discretize_state(self, observation):
        """Convert continuous state to discrete state indices"""
        position, velocity = observation
        
        # Find closest bin indices
        pos_idx = np.digitize(position, self.position_space) - 1
        vel_idx = np.digitize(velocity, self.velocity_space) - 1
        
        # Clamp to valid range
        pos_idx = max(0, min(pos_idx, self.position_bins - 1))
        vel_idx = max(0, min(vel_idx, self.velocity_bins - 1))
        
        return (pos_idx, vel_idx)
    
    def get_action(self, state, training=True):
        """Epsilon-greedy action selection"""
        discrete_state = self.discretize_state(state)
        
        if training and np.random.random() < self.epsilon:
            return np.random.randint(0, self.n_actions)
        else:
            return np.argmax(self.q_table[discrete_state])
    
    def update_q_table(self, state, action, reward, next_state, done):
        """Update Q-table using Q-learning update rule"""
        discrete_state = self.discretize_state(state)
        discrete_next_state = self.discretize_state(next_state)
        
        # Current Q-value
        current_q = self.q_table[discrete_state][action]
        
        # Next state's max Q-value
        if done:
            max_next_q = 0
        else:
            max_next_q = np.max(self.q_table[discrete_next_state])
        
        # Q-learning update
        new_q = current_q + self.lr * (reward + self.gamma * max_next_q - current_q)
        self.q_table[discrete_state][action] = new_q
    
    def decay_epsilon(self):
        """Decay exploration rate"""
        self.epsilon = max(self.min_epsilon, self.epsilon * self.epsilon_decay)

def train_q_learning(episodes=1000, max_steps=200):
    """Train Q-learning agent"""
    env = gym.make('MountainCar-v0')
    agent = QLearningAgent()
    
    rewards_per_episode = []
    steps_per_episode = []
    successes = []
    
    for episode in range(episodes):
        state, _ = env.reset()
        total_reward = 0
        
        for step in range(max_steps):
            action = agent.get_action(state, training=True)
            next_state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated
            
            agent.update_q_table(state, action, reward, next_state, done)
            
            state = next_state
            total_reward += reward
            
            if done:
                success = state[0] >= 0.5
                successes.append(success)
                break
        else:
            successes.append(False)
        
        agent.decay_epsilon()
        rewards_per_episode.append(total_reward)
        steps_per_episode.append(step + 1)
        
        # Print progress
        if (episode + 1) % 200 == 0:
            recent_success_rate = np.mean(successes[-200:]) if len(successes) >= 200 else np.mean(successes)
            avg_steps = np.mean(steps_per_episode[-200:]) if len(steps_per_episode) >= 200 else np.mean(steps_per_episode)
            print(f"Episode {episode + 1}: Success rate = {recent_success_rate:.2%}, Avg steps = {avg_steps:.1f}, Epsilon = {agent.epsilon:.3f}")
    
    env.close()
    return agent, rewards_per_episode, steps_per_episode, successes

In [ ]:
# Train the Q-learning agent
print("Training Q-learning agent...")
q_agent, rewards, steps, successes = train_q_learning(episodes=1000)

In [ ]:
# Test the trained Q-learning agent
def test_q_agent(agent, num_episodes=10):
    env = gym.make('MountainCar-v0')
    
    total_rewards = []
    episode_lengths = []
    successes = 0
    
    for episode in range(num_episodes):
        observation, info = env.reset()
        total_reward = 0
        
        for step in range(200):
            action = agent.get_action(observation, training=False)
            observation, reward, terminated, truncated, info = env.step(action)
            total_reward += reward
            
            if terminated or truncated:
                if observation[0] >= 0.5:
                    successes += 1
                break
        
        total_rewards.append(total_reward)
        episode_lengths.append(step + 1)
        
        if num_episodes <= 5:
            print(f"Episode {episode + 1}: Reward = {total_reward}, Steps = {step + 1}, Success = {observation[0] >= 0.5}")
    
    env.close()
    
    avg_reward = np.mean(total_rewards)
    avg_steps = np.mean(episode_lengths)
    success_rate = successes / num_episodes
    
    print(f"\nQ-Learning Results over {num_episodes} episodes:")
    print(f"Average reward: {avg_reward:.2f}")
    print(f"Average steps: {avg_steps:.2f}")
    print(f"Success rate: {success_rate:.2%}")
    
    return avg_reward, avg_steps, success_rate

# Test the trained agent
test_q_agent(q_agent, num_episodes=5)

## Training Progress Visualization

In [ ]:
# Plot training progress
def plot_training_progress(rewards, steps, successes, window=100):
    fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(15, 10))
    
    # Moving average for smoother plots
    def moving_average(data, window):
        return np.convolve(data, np.ones(window)/window, mode='valid')
    
    episodes = range(len(rewards))
    
    # Rewards
    ax1.plot(episodes, rewards, alpha=0.3, color='blue', label='Episode Reward')
    if len(rewards) >= window:
        smooth_rewards = moving_average(rewards, window)
        ax1.plot(range(window-1, len(rewards)), smooth_rewards, color='red', label=f'{window}-Episode Average')
    ax1.set_xlabel('Episode')
    ax1.set_ylabel('Total Reward')
    ax1.set_title('Training Rewards')
    ax1.legend()
    ax1.grid(True)
    
    # Steps per episode
    ax2.plot(episodes, steps, alpha=0.3, color='green', label='Episode Steps')
    if len(steps) >= window:
        smooth_steps = moving_average(steps, window)
        ax2.plot(range(window-1, len(steps)), smooth_steps, color='orange', label=f'{window}-Episode Average')
    ax2.set_xlabel('Episode')
    ax2.set_ylabel('Steps to Completion')
    ax2.set_title('Training Steps')
    ax2.legend()
    ax2.grid(True)
    
    # Success rate
    success_rate = []
    for i in range(len(successes)):
        if i < window:
            success_rate.append(np.mean(successes[:i+1]))
        else:
            success_rate.append(np.mean(successes[i-window+1:i+1]))
    
    ax3.plot(episodes, success_rate, color='purple', label=f'Success Rate ({window}-episode window)')
    ax3.set_xlabel('Episode')
    ax3.set_ylabel('Success Rate')
    ax3.set_title('Training Success Rate')
    ax3.legend()
    ax3.grid(True)
    ax3.set_ylim(-0.05, 1.05)
    
    # Epsilon decay
    # Reconstruct epsilon values (approximation)
    epsilon_values = []
    epsilon = 1.0
    epsilon_decay = 0.995
    min_epsilon = 0.01
    
    for _ in range(len(rewards)):
        epsilon_values.append(epsilon)
        epsilon = max(min_epsilon, epsilon * epsilon_decay)
    
    ax4.plot(episodes, epsilon_values, color='brown', label='Epsilon (Exploration Rate)')
    ax4.set_xlabel('Episode')
    ax4.set_ylabel('Epsilon')
    ax4.set_title('Exploration Rate Decay')
    ax4.legend()
    ax4.grid(True)
    
    plt.tight_layout()
    plt.show()

# Plot the results
plot_training_progress(rewards, steps, successes)

## Visualization Demo

Run this cell to see the trained agent in action (requires display).

In [ ]:
def visualize_trained_agent(agent, episodes=3):
    """Visualize the trained agent solving MountainCar"""
    env = gym.make('MountainCar-v0', render_mode='human')
    
    for episode in range(episodes):
        observation, _ = env.reset()
        total_reward = 0
        print(f"\nEpisode {episode + 1}:")
        
        for step in range(200):
            action = agent.get_action(observation, training=False)
            observation, reward, terminated, truncated, _ = env.step(action)
            total_reward += reward
            
            # Print some steps for debugging
            if step % 20 == 0 or step < 10:
                pos, vel = observation
                action_names = ['Left', 'None', 'Right']
                print(f"  Step {step:3d}: Pos={pos:+.3f}, Vel={vel:+.4f}, Action={action_names[action]}")
            
            if terminated or truncated:
                success = observation[0] >= 0.5
                print(f"  Finished in {step + 1} steps. Success: {success}")
                print(f"  Final position: {observation[0]:.3f}, Final velocity: {observation[1]:.4f}")
                print(f"  Total reward: {total_reward}")
                break
            
            time.sleep(0.05)  # Small delay for better visualization
    
    env.close()

# Uncomment to visualize (requires display)
# visualize_trained_agent(q_agent, episodes=2)

## Policy Comparison

Let's compare different approaches to solving MountainCar.

In [ ]:
def random_policy(observation):
    """Random action selection"""
    return np.random.randint(0, 3)

def position_based_policy(observation):
    """Simple position-based policy"""
    position, velocity = observation
    
    # If far left, go right; if far right, go left; otherwise follow velocity
    if position < -0.5:
        return 2  # Go right
    elif position > 0.3:
        return 0  # Go left
    else:
        return 2 if velocity >= 0 else 0

# Compare different policies
policies = {
    'Random': random_policy,
    'Momentum': momentum_policy,
    'Position-based': position_based_policy,
    'Q-Learning': lambda obs: q_agent.get_action(obs, training=False)
}

print("Policy Comparison (50 episodes each):\n")
results = {}

for name, policy in policies.items():
    print(f"Testing {name} Policy:")
    avg_reward, avg_steps, success_rate = test_policy(policy, num_episodes=50, render_mode=None)
    results[name] = {'avg_reward': avg_reward, 'avg_steps': avg_steps, 'success_rate': success_rate}
    print()

# Summary comparison
print("\n" + "="*70)
print("SUMMARY COMPARISON")
print("="*70)
print(f"{'Policy':<15} {'Avg Reward':<12} {'Avg Steps':<12} {'Success Rate':<15}")
print("-"*70)

for name, result in results.items():
    print(f"{name:<15} {result['avg_reward']:<12.2f} {result['avg_steps']:<12.1f} {result['success_rate']:<15.1%}")

## Key Insights

1. **Momentum Strategy**: The key to solving MountainCar is building momentum by oscillating back and forth
2. **Q-Learning**: Can learn an optimal policy through trial and error, typically outperforming hand-crafted policies
3. **State Discretization**: Continuous spaces need to be discretized for tabular methods like Q-learning
4. **Exploration vs Exploitation**: Epsilon-greedy strategy balances learning new actions vs using known good actions

## Further Improvements

- **Deep Q-Learning (DQN)**: Use neural networks instead of tables for better generalization
- **Policy Gradient Methods**: Learn policies directly instead of value functions
- **Actor-Critic Methods**: Combine value-based and policy-based approaches
- **Continuous Control**: Use algorithms like DDPG or SAC for continuous action spaces
